# TASK03B — SAM 3D Body → MHR → clad-body: isolated dual-environment Colab run

**Technical smoke test only. Not an anthropometric accuracy benchmark.**
See `docs/experiments/TASK03B_DEPENDENCY_RESOLUTION.md`.

A real Colab run of the Task 03 notebook hit `DEPENDENCY_ENVIRONMENT_BLOCKED` /
`CUDA_PYTORCH_MISMATCH`: GPU, HF auth, and checkpoint download all worked, but the shared
main-kernel install (SAM 3D Body's own deps installed alongside everything else, with no
protection against a transitive dependency silently changing Colab's pre-matched
torch/torchvision/CUDA-driver triplet) broke before inference could run.

This version uses **two fully isolated venvs**, neither of which touches Colab's ambient
kernel packages:
- **Environment A** (`/content/env_sam3d`) — SAM 3D Body GPU inference only. Pinned to the
  *exact* torch/torchvision versions Colab's ambient kernel already has working with this
  runtime's GPU driver (read once, then protected — never left to a transitive pip resolve).
- **Environment B** (`/content/MHR`, Pixi-managed, pip-venv fallback) — MHR + clad-body
  measurement extraction, CPU-only, no GPU/CUDA dependency at all.

They exchange exactly one small, versioned, inspectable file
(`experiments/sam3d_mhr_clad_smoke/interchange.py`) — never a Python object, torch tensor,
or pickle.

Requirements: GPU runtime (`Runtime > Change runtime type`), and a Colab Secret named
exactly `HF_TOKEN` with approved access to `facebook/sam-3d-body-dinov3` and
`facebook/sam-3d-body-vith`. Run top-to-bottom.

## 1. Environment inspection

In [ ]:
import platform, shutil, subprocess, sys, time, os

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

def run_shell(cmd, timeout=1800, cwd=None):
    print(f'$ {cmd}')
    proc = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout, cwd=cwd)
    if proc.returncode != 0:
        print(proc.stdout[-3000:])
        print(proc.stderr[-3000:])
    return proc.returncode == 0

print('Python:', platform.python_version())
print('Platform:', platform.platform())

try:
    import psutil
    ram_gb = psutil.virtual_memory().total / 1e9
except ImportError:
    ram_gb = None
print('System RAM (GB):', round(ram_gb, 1) if ram_gb else 'unknown')

disk = shutil.disk_usage('/')
print(f'Disk: {disk.free/1e9:.1f} GB free / {disk.total/1e9:.1f} GB total')

gpu_query = sh('nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader 2>/dev/null')
gpu_available = bool(gpu_query)
print('nvidia-smi GPU query:', gpu_query if gpu_available else 'NONE DETECTED')

# Capture the AMBIENT Colab kernel's already-GPU-working torch/torchvision as the pin
# reference for Environment A -- Colab guarantees this pair matches this runtime's driver;
# we do not guess a version, we reproduce this exact one inside an isolated venv (section 4).
ambient_torch_version = None
ambient_torchvision_version = None
ambient_cuda_version = None
try:
    import torch
    ambient_torch_version = torch.__version__.split('+')[0]
    ambient_cuda_version = torch.version.cuda
    print('Ambient torch:', torch.__version__, '| CUDA build:', ambient_cuda_version,
          '| torch.cuda.is_available():', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('CUDA device:', torch.cuda.get_device_name(0))
        print('VRAM total (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
    else:
        gpu_available = False
    try:
        import torchvision
        ambient_torchvision_version = torchvision.__version__.split('+')[0]
        print('Ambient torchvision:', torchvision.__version__)
    except ImportError:
        print('Ambient torchvision: not installed (Environment A will pick a compatible one)')
except ImportError:
    print('torch not importable in ambient kernel')
    gpu_available = False

if not gpu_available:
    raise RuntimeError(
        'NO_GPU: no usable NVIDIA GPU detected. Go to Runtime > Change runtime type, select a '
        'GPU, then Runtime > Restart and run all. This notebook does not attempt full '
        'inference on CPU (Task 03 section 1).'
    )

## 2. Secure HF_TOKEN retrieval

Colab Secrets only. Never printed, logged, written to a file, or passed to
`huggingface_hub.login()` (persists to disk). Only ever used as a `token=` argument, and only
from the ambient kernel -- `google.colab.userdata` is not reachable from a subprocess venv,
which is exactly why checkpoint download (needs the token) stays in the ambient kernel while
Environment A (needs no token, just the already-downloaded files) runs as an isolated subprocess.

In [ ]:
from google.colab import userdata

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception as exc:
    raise RuntimeError(
        "HF_AUTH_FAILURE: could not read the 'HF_TOKEN' Colab Secret. Add it via the key icon "
        "in the left sidebar, name it exactly HF_TOKEN, and enable notebook access. "
        f"Original error: {exc!r}"
    )
if not HF_TOKEN:
    raise RuntimeError('HF_AUTH_FAILURE: HF_TOKEN secret is empty.')
print(f'HF_TOKEN retrieved from Colab Secrets: OK ({len(HF_TOKEN)} chars, value not shown)')

## 3. Repository checkout and shared (torch-free) helper imports

In [ ]:
REPO_URL = 'https://github.com/eliyahumines-dot/mtm-body-checker.git'
REPO_BRANCH = 'claude/body-measurement-feasibility-4qgdrz'  # update once merged to main
REPO_DIR = '/content/mtm-body-checker'

if not os.path.isdir(REPO_DIR):
    run_shell(f'git clone --branch {REPO_BRANCH} --depth 1 {REPO_URL} {REPO_DIR}')
else:
    print('Repo already present at', REPO_DIR)

EXPERIMENT_DIR = f'{REPO_DIR}/experiments/sam3d_mhr_clad_smoke'
if not os.path.isdir(EXPERIMENT_DIR):
    raise RuntimeError(
        f'{EXPERIMENT_DIR} not found. If this repo is private and the clone failed, manually '
        f'upload experiments/sam3d_mhr_clad_smoke/ to {EXPERIMENT_DIR} and re-run this cell.'
    )
sys.path.insert(0, EXPERIMENT_DIR)
print('Reusing Task 02/03/03B code from:', EXPERIMENT_DIR)

# All of these are plain Python + numpy -- none require torch, so they're safe to import into
# the ambient kernel even though the ambient kernel never runs SAM 3D Body or clad-body itself.
from decision_gate import PipelineState, FailureCategory, classify, phase_summary
from interchange import read_interchange, interchange_to_clad_params, InterchangeError
from run import measure_via_subprocess

state = PipelineState(gpu_available=gpu_available)
WORK_DIR = '/content/work'
os.makedirs(WORK_DIR, exist_ok=True)

# PHASE A — SAM 3D Body GPU environment

Everything in this phase runs inside `/content/env_sam3d`, a dedicated venv, invoked only via
subprocess. Colab's ambient kernel packages are never modified.

## 4. Build Environment A (pinned torch/torchvision, SAM 3D Body deps, Detectron2)

In [ ]:
ENV_A_DIR = '/content/env_sam3d'
env_a_python = f'{ENV_A_DIR}/bin/python3'

# Prefer Python 3.11 (upstream SAM 3D Body's documented target) if available or installable;
# fall back to whatever python3 is present rather than hard-failing on this alone.
python311 = shutil.which('python3.11')
if python311 is None:
    run_shell('apt-get -qq update && apt-get -qq install -y python3.11 python3.11-venv python3.11-dev')
    python311 = shutil.which('python3.11')
venv_builder = python311 or sys.executable
print('Building Environment A with:', venv_builder)

sam3d_environment_ok = run_shell(f'{venv_builder} -m venv {ENV_A_DIR}')
sam3d_environment_ok &= run_shell(f'{env_a_python} -m pip install -q --upgrade pip')

# Pin torch/torchvision to the ambient kernel's own already-GPU-working versions -- do not
# guess a version; reproduce the one Colab already proved compatible with this runtime's driver.
torch_pin = f'torch=={ambient_torch_version}' if ambient_torch_version else 'torch'
if ambient_torchvision_version:
    torch_pin += f' torchvision=={ambient_torchvision_version}'
sam3d_environment_ok &= run_shell(f'{env_a_python} -m pip install -q {torch_pin}')

pin_check = sh(f'{env_a_python} -c "import torch; print(torch.__version__)"')
print('Environment A torch after pin:', pin_check)
if ambient_torch_version and ambient_torch_version not in pin_check:
    print(f'WARNING: resolved {pin_check!r}, not the pinned {ambient_torch_version!r} -- '
          f'CUDA/driver compatibility is not guaranteed for this run.')

In [ ]:
SAM3D_DIR = '/content/sam-3d-body'
if not os.path.isdir(SAM3D_DIR):
    sam3d_environment_ok &= run_shell(f'git clone --depth 1 https://github.com/facebookresearch/sam-3d-body.git {SAM3D_DIR}')

sam3d_pip_deps = (
    'pytorch-lightning pyrender opencv-python yacs scikit-image einops timm dill pandas rich '
    'hydra-core hydra-submitit-launcher hydra-colorlog pyrootutils webdataset chumpy '
    'networkx==3.2.1 roma joblib seaborn wandb appdirs jsonlines xtcocotools loguru optree '
    'fvcore pycocotools huggingface_hub numpy'
)
sam3d_environment_ok &= run_shell(f'{env_a_python} -m pip install -q {sam3d_pip_deps}')
sam3d_environment_ok &= run_shell(
    f"{env_a_python} -m pip install -q 'git+https://github.com/facebookresearch/detectron2.git@a1ce2f9' "
    f"--no-build-isolation --no-deps"
)
sam3d_environment_ok &= run_shell(f'{env_a_python} -m pip install -q git+https://github.com/microsoft/MoGe.git --no-deps')

# Verify the torch/torchvision pin actually survived every install above -- this is the
# specific check that would have caught Task 03's CUDA_PYTORCH_MISMATCH before it reached
# inference, by refusing to proceed on a silently-changed pin instead of finding out later.
post_check = sh(f'{env_a_python} -c "import torch, torchvision; print(torch.__version__, torchvision.__version__)"')
print('Environment A torch/torchvision after full install:', post_check)
pin_violated = bool(ambient_torch_version) and ambient_torch_version not in post_check
if pin_violated:
    sam3d_environment_ok = False
    state.add_failure(FailureCategory.CUDA_PYTORCH_MISMATCH)
    print('CUDA_PYTORCH_MISMATCH: a dependency silently changed the pinned torch/torchvision version.')

state.dependencies_installed = sam3d_environment_ok
print()
print('PHASE A environment build:', 'PASS' if sam3d_environment_ok else 'FAIL')

## 5. Checkpoint authentication and download

Confirmed working in the human's real run (GPU/HF-auth/checkpoint-download all passed) — kept
essentially unchanged from Task 03, per this task's instruction not to re-investigate this area.

In [ ]:
from huggingface_hub import HfApi, snapshot_download

CHECKPOINT_REPOS = ['facebook/sam-3d-body-dinov3', 'facebook/sam-3d-body-vith']
PRIMARY_CHECKPOINT_REPO = 'facebook/sam-3d-body-dinov3'  # upstream README's own default example

api = HfApi()
access_status = {}
for repo_id in CHECKPOINT_REPOS:
    try:
        api.model_info(repo_id, token=HF_TOKEN)
        access_status[repo_id] = 'accessible'
    except Exception as exc:
        access_status[repo_id] = f'BLOCKED: {type(exc).__name__}'
print(access_status)

hf_auth_ok = access_status.get(PRIMARY_CHECKPOINT_REPO) == 'accessible'
state.hf_auth_ok = hf_auth_ok
if not hf_auth_ok:
    state.add_failure(FailureCategory.HF_AUTH_FAILURE)
    raise RuntimeError(f'CHECKPOINT_ACCESS_BLOCKED: access to {PRIMARY_CHECKPOINT_REPO} not confirmed.')

CKPT_DIR = f"/content/checkpoints/{PRIMARY_CHECKPOINT_REPO.split('/')[-1]}"
checkpoint_downloaded = False
checkpoint_download_time_s = None
checkpoint_size_bytes = None
try:
    t0 = time.time()
    snapshot_download(repo_id=PRIMARY_CHECKPOINT_REPO, local_dir=CKPT_DIR, token=HF_TOKEN)
    checkpoint_download_time_s = round(time.time() - t0, 1)
    checkpoint_downloaded = True
    checkpoint_size_bytes = int(sh(f'du -sb {CKPT_DIR}').split()[0])
    print(f'Downloaded {PRIMARY_CHECKPOINT_REPO} in {checkpoint_download_time_s}s, {checkpoint_size_bytes} bytes')
except Exception as exc:
    print('CHECKPOINT_ACCESS_FAILURE:', repr(exc))
    state.add_failure(FailureCategory.CHECKPOINT_ACCESS_FAILURE)

state.checkpoint_downloaded = checkpoint_downloaded
if not checkpoint_downloaded:
    raise RuntimeError('CHECKPOINT_ACCESS_BLOCKED: checkpoint download failed after confirmed access.')

## 6. Input image selection

Defaults to a public sample bundled in the official `sam-3d-body` repo. Set
`USE_SAMPLE_IMAGE = False` to upload your own photo instead — stays only in this Colab
runtime's `/content`, never written into the cloned git repo.

In [ ]:
USE_SAMPLE_IMAGE = True
SAMPLE_IMAGE = f'{SAM3D_DIR}/assets/qualitative_comparisons/sample1/input_bbox.png'

if USE_SAMPLE_IMAGE:
    input_image_path = SAMPLE_IMAGE
    used_sample_image = True
    if not os.path.exists(input_image_path):
        raise RuntimeError(f'Bundled sample image not found at {input_image_path}.')
else:
    from google.colab import files
    uploaded = files.upload()
    input_image_path = f'/content/{next(iter(uploaded))}'
    used_sample_image = False

print('Input image:', input_image_path, '(public sample)' if used_sample_image else '(user-uploaded, not committed)')

## 7. Real SAM 3D Body inference (Environment A subprocess) → interchange file

Runs `_sam3d_inference_worker.py` inside Environment A's venv. Captures actual output schema,
`mhr_model_params` shape, load/inference time, and peak VRAM — nothing here is assumed from docs.

In [ ]:
def run_phase_a_inference(checkpoint_dir, image_path, work_dir=WORK_DIR, tag='primary'):
    interchange_path = f'{work_dir}/{tag}_interchange.npz'
    telemetry_path = f'{work_dir}/{tag}_telemetry.json'
    cmd = [env_a_python, f'{EXPERIMENT_DIR}/_sam3d_inference_worker.py',
           image_path, checkpoint_dir, interchange_path, telemetry_path]
    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=900)
    if os.path.exists(telemetry_path):
        with open(telemetry_path) as f:
            telemetry = json.load(f)
    else:
        telemetry = {'status': 'error', 'error': 'worker produced no telemetry file'}
    telemetry['returncode'] = proc.returncode
    telemetry['interchange_path'] = interchange_path if telemetry.get('interchange_written') else None
    if proc.returncode != 0 and not telemetry.get('error'):
        telemetry['error'] = f'worker exited {proc.returncode}'
    if proc.returncode != 0:
        telemetry['stderr_tail'] = (proc.stderr or '')[-1500:]
    return telemetry

import json

phase_a_ok = False
if sam3d_environment_ok:
    primary_telemetry = run_phase_a_inference(CKPT_DIR, input_image_path)
    print(json.dumps({k: v for k, v in primary_telemetry.items() if k != 'output_schema'}, indent=2))
    phase_a_ok = primary_telemetry.get('status') == 'ok'
    state.sam3d_inference_ok = primary_telemetry.get('status') in ('ok', 'ok_no_person_detected')
    state.mhr_schema_valid = primary_telemetry.get('interchange_written', False)
    if not state.sam3d_inference_ok:
        state.add_failure(FailureCategory.SAM3D_INFERENCE_FAILURE)
    if state.sam3d_inference_ok and not state.mhr_schema_valid:
        state.add_failure(FailureCategory.MHR_SCHEMA_FAILURE)
else:
    primary_telemetry = {'status': 'skipped', 'error': 'Environment A failed to build (see section 4)'}
    print('Skipping SAM 3D Body inference -- Environment A did not build successfully.')

print()
print('PHASE A — SAM3D GPU inference:', 'PASS' if phase_a_ok else 'FAIL')

### Actual SAM 3D Body output schema (as observed, not assumed from docs)

In [ ]:
for k, v in primary_telemetry.get('output_schema', {}).items():
    print(f"  {k}: shape={v['shape']} dtype={v['dtype']}")
if not primary_telemetry.get('output_schema'):
    print('No output schema captured.')

mp = primary_telemetry.get('output_schema', {}).get('mhr_model_params')
sp = primary_telemetry.get('output_schema', {}).get('shape_params')
scp = primary_telemetry.get('output_schema', {}).get('scale_params')
print()
print('mhr_model_params shape:', mp['shape'] if mp else 'n/a', '(expected [204])')
print('shape_params shape:', sp['shape'] if sp else 'n/a', '(expected [45])')
if scp:
    print('scale_params shape (NOT written to the interchange file -- see adapter.py docstring):', scp['shape'])

# PHASE B — MHR + clad-body measurement extraction

Runs only if Phase A produced an interchange file. CPU-only, no GPU/CUDA dependency. Built and
validated **independently** of Phase A, in this order (Task 03B section 4): (1) import
pymomentum, (2) load official MHR assets, (3) reconstruct an MHR body from a **known-good
bundled fixture**, (4) run clad-body measurement extraction on it — all before step (5), feeding
the real Phase A output through. This separates an environment failure from a SAM3D→MHR schema
failure.

## 8. Build Environment B

Prefers the officially-recommended Pixi installation path (MHR's own README calls this "the
most reliable environment setup"; Task 02's plain-pip `pymomentum-cpu` install segfaulted).
Falls back automatically to a pip-based CPU-torch venv (Task 02/03's approach) only if Pixi
itself fails to install or resolve — the human never has to choose manually.

In [ ]:
MHR_REPO_DIR = '/content/MHR'
env_b_python = None
mhr_clad_env_build_ok = False

# --- Primary path: Pixi (MHR's own documented recommended installation method) ---
pixi_ok = run_shell('curl -fsSL https://pixi.sh/install.sh | bash')
pixi_bin = os.path.expanduser('~/.pixi/bin/pixi')
pixi_ok = pixi_ok and os.path.exists(pixi_bin)

if pixi_ok and not os.path.isdir(MHR_REPO_DIR):
    pixi_ok &= run_shell(f'git clone --depth 1 https://github.com/facebookresearch/MHR.git {MHR_REPO_DIR}')

if pixi_ok:
    pixi_ok &= run_shell(f'{pixi_bin} install --manifest-path {MHR_REPO_DIR}/pixi.toml', timeout=1200)
if pixi_ok:
    pixi_ok &= run_shell(
        f'{pixi_bin} run --manifest-path {MHR_REPO_DIR}/pixi.toml download-assets', timeout=900
    )
candidate_env_b_python = f'{MHR_REPO_DIR}/.pixi/envs/default/bin/python'
if pixi_ok and os.path.exists(candidate_env_b_python):
    pixi_ok &= run_shell(f'{candidate_env_b_python} -m pip install -q --no-deps clad-body')
if pixi_ok and os.path.exists(candidate_env_b_python):
    env_b_python = candidate_env_b_python
    mhr_clad_env_build_ok = True
    print('Environment B built via Pixi at', env_b_python)
else:
    print('Pixi path did not complete successfully -- falling back to a pip-based CPU-torch venv',
          '(Task 02/03 approach; documented as experimental by MHR upstream).')

In [ ]:
# --- Fallback path: pip + CPU-only torch venv (only runs if Pixi path above did not succeed) ---
if not mhr_clad_env_build_ok:
    ENV_B_FALLBACK_DIR = '/content/env_mhr_clad_fallback'
    fallback_python = f'{ENV_B_FALLBACK_DIR}/bin/python3'
    ok = run_shell(f'python3 -m venv {ENV_B_FALLBACK_DIR}')
    ok &= run_shell(f'{fallback_python} -m pip install -q --upgrade pip')
    ok &= run_shell(f'{fallback_python} -m pip install -q torch --index-url https://download.pytorch.org/whl/cpu')
    ok &= run_shell(f"{fallback_python} -m pip install -q 'clad-body[mhr]'")
    if ok:
        import glob
        site_pkgs = glob.glob(f'{ENV_B_FALLBACK_DIR}/lib/python*/site-packages')
        assets_dir = f'{site_pkgs[0]}/assets' if site_pkgs else None
        if assets_dir and not os.path.isdir(assets_dir):
            run_shell(f'mkdir -p {assets_dir}')
            ok &= run_shell('curl -sSL -o /content/mhr_assets.zip '
                            'https://github.com/facebookresearch/MHR/releases/latest/download/assets.zip')
            ok &= run_shell(f'unzip -q -o /content/mhr_assets.zip -d {assets_dir}')
            nested = f'{assets_dir}/assets'
            if os.path.isdir(nested):
                run_shell(f'mv {nested}/* {assets_dir}/ && rmdir {nested}')
    if ok:
        env_b_python = fallback_python
        mhr_clad_env_build_ok = True
        print('Environment B built via pip fallback at', env_b_python)
    else:
        print('Both the Pixi path and the pip fallback failed to build Environment B.')

print()
print('PHASE B environment build:', 'PASS' if mhr_clad_env_build_ok else 'FAIL')

## 9. Environment B self-test (steps 1-4): import pymomentum, load MHR assets, reconstruct + measure a bundled fixture -- before touching any real SAM3D output

In [ ]:
def run_phase_b_selftest(python_exe):
    check_script = (
        "import pymomentum.geometry\n"
        "from mhr.mhr import MHR\n"
        "MHR.from_files(device='cpu', wants_pose_correctives=False)\n"
        "print('OK')\n"
    )
    proc = subprocess.run([python_exe, '-c', check_script], capture_output=True, text=True, timeout=300)
    if proc.returncode != 0:
        return {'ok': False, 'stage': 'import_pymomentum_or_load_mhr_assets',
                'returncode': proc.returncode, 'stderr_tail': (proc.stderr or '')[-1500:]}

    probe = subprocess.run(
        [python_exe, '-c', 'import clad_body, os; print(os.path.dirname(clad_body.__file__))'],
        capture_output=True, text=True, timeout=60,
    )
    if probe.returncode != 0:
        return {'ok': False, 'stage': 'locate_clad_body_package',
                'returncode': probe.returncode, 'stderr_tail': (probe.stderr or '')[-1500:]}
    fixture_path = f'{probe.stdout.strip()}/measure/testdata/mhr/female_average/mhr_params.json'
    if not os.path.exists(fixture_path):
        return {'ok': False, 'stage': 'bundled_fixture_not_found', 'fixture_path': fixture_path}

    w, fl = [], []
    status, result = measure_via_subprocess(
        fixture_path, known_height_cm=None, warnings=w, failures=fl, python_executable=python_exe,
    )
    return {'ok': status == 'ok', 'stage': 'reconstruct_and_measure_bundled_fixture',
            'status': status, 'warnings': w, 'failures': fl, 'result': result}

mhr_clad_environment_ok = False
if mhr_clad_env_build_ok:
    selftest = run_phase_b_selftest(env_b_python)
    print(json.dumps({k: v for k, v in selftest.items() if k != 'result'}, indent=2))
    mhr_clad_environment_ok = selftest['ok']
    if not mhr_clad_environment_ok:
        stage = selftest.get('stage', '')
        state.add_failure(
            FailureCategory.PYMOMENTUM_FAILURE if 'pymomentum' in stage or 'mhr_assets' in stage
            else FailureCategory.CLAD_BODY_FAILURE
        )
else:
    selftest = {'ok': False, 'stage': 'environment_build_failed'}
    print('Skipping self-test -- Environment B did not build.')

state.mhr_clad_environment_ok = mhr_clad_environment_ok
print()
print('PHASE B self-test (steps 1-4, bundled fixture):', 'PASS' if mhr_clad_environment_ok else 'FAIL')

## 10-11. Step 5: real SAM3D interchange data → MHR reconstruction → clad-body measurements

Only reached if both Phase A produced an interchange file AND Phase B's self-test passed --
this is what separates an environment failure from a genuine SAM3D→MHR schema failure
(Task 03B section 4).

In [ ]:
import tempfile

def run_phase_b_extraction(interchange_path, python_exe, known_height_cm=None):
    record = read_interchange(interchange_path)  # raises InterchangeError on a malformed/mismatched file
    clad_params = interchange_to_clad_params(record)
    with tempfile.NamedTemporaryFile(mode='w', suffix='_sam3d_mhr_params.json', delete=False) as f:
        json.dump(clad_params, f)
        params_path = f.name
    w, fl = [], []
    status, raw_res = measure_via_subprocess(
        params_path, known_height_cm=None, warnings=w, failures=fl, python_executable=python_exe,
    )
    calibrated_res = None
    if status == 'ok' and known_height_cm is not None:
        w2, fl2 = [], []
        _, calibrated_res = measure_via_subprocess(
            params_path, known_height_cm=known_height_cm, warnings=w2, failures=fl2, python_executable=python_exe,
        )
        w.extend(w2); fl.extend(fl2)
    try:
        os.unlink(params_path)
    except OSError:
        pass
    return {'status': status, 'raw_result': raw_res, 'calibrated_result': calibrated_res,
            'warnings': w, 'failures': fl, 'record': record}

KNOWN_HEIGHT_CM = None  # e.g. 178.0 -- customer-reported height in cm, or None to skip calibration

extraction = None
if phase_a_ok and mhr_clad_environment_ok and primary_telemetry.get('interchange_path'):
    try:
        extraction = run_phase_b_extraction(
            primary_telemetry['interchange_path'], env_b_python, known_height_cm=KNOWN_HEIGHT_CM,
        )
    except InterchangeError as exc:
        extraction = {'status': f'interchange_error: {exc}', 'raw_result': None, 'calibrated_result': None,
                      'warnings': [], 'failures': [str(exc)]}
    print('measurement_extraction_status:', extraction['status'])
    for w in extraction['warnings']:
        print('WARNING:', w)
    for f_ in extraction['failures']:
        print('FAILURE:', f_)
    state.mhr_reconstruction_ok = extraction['status'] != 'blocked_native_crash'
    state.clad_body_measure_ok = extraction['status'] == 'ok'
    if extraction['status'] == 'ok':
        state.measurements = extraction['raw_result']['measurements_cm']
    else:
        state.add_failure(FailureCategory.CLAD_BODY_FAILURE)
else:
    print('Skipping step 5 -- Phase A and/or Phase B self-test did not pass; see PASS/FAIL above.')

raw_measurements_cm = extraction['raw_result']['measurements_cm'] if extraction and extraction['raw_result'] else None
raw_body_height_cm = extraction['raw_result']['raw_body_height_cm'] if extraction and extraction['raw_result'] else None
calibrated_measurements_cm = extraction['calibrated_result']['measurements_cm'] if extraction and extraction['calibrated_result'] else None
calibrated_scale_factor = extraction['calibrated_result']['rescale_factor'] if extraction and extraction['calibrated_result'] else None

print()
print('RAW body height (cm):', raw_body_height_cm)
print('RAW measurements (cm):', json.dumps(raw_measurements_cm, indent=2) if raw_measurements_cm else None)
if calibrated_measurements_cm:
    print(f'scale_factor = known_height / predicted_height = {KNOWN_HEIGHT_CM} / {raw_body_height_cm:.2f} = {calibrated_scale_factor:.4f}')
    print('CALIBRATED measurements (cm):', json.dumps(calibrated_measurements_cm, indent=2))
print()
print('PHASE B extraction (step 5, real SAM3D data):', 'PASS' if (extraction and extraction['status'] == 'ok') else 'FAIL')

## 12. Known-height calibration — note

Already applied above via `KNOWN_HEIGHT_CM` (two independent worker calls: one with
`known_height_cm=None` for the raw result, one with it set for the calibrated result — never
computed by mutating the raw result). The scale factor is a single uniform multiplier
(`known_height_cm / raw_body_height_cm`) applied to every mesh vertex; it corrects overall
scale only and does **not** alter body proportions to match any customer-reported measurement
other than height. See `rescale.py`'s docstring for what it does not correct (camera
perspective, posture, non-uniform proportion errors).

### MTM measurement terminology mapping (reused from Task 02, unchanged)

In [ ]:
from mtm_mapping import MTM_MEASUREMENT_MAP

if raw_measurements_cm:
    for m in MTM_MEASUREMENT_MAP:
        val = raw_measurements_cm.get(m.clad_body_key) if m.clad_body_key else None
        print(f'{m.mtm_name:32s} <- {str(m.clad_body_key):20s} = {val}  [{m.confidence}]')
else:
    print('No raw measurements available to map.')

## 13. Save results

In [ ]:
import datetime, pathlib

RESULTS_DIR_RUNTIME = '/content/results'
os.makedirs(RESULTS_DIR_RUNTIME, exist_ok=True)

output_record = {
    'subject_id': pathlib.Path(input_image_path).stem,
    'used_public_sample_image': used_sample_image,
    'checkpoint_used': PRIMARY_CHECKPOINT_REPO if checkpoint_downloaded else None,
    'checkpoint_size_bytes': checkpoint_size_bytes,
    'phase_a_status': primary_telemetry.get('status'),
    'person_detected': primary_telemetry.get('person_detected'),
    'sam3d_output_schema': primary_telemetry.get('output_schema'),
    'phase_b_environment_selftest': {k: v for k, v in selftest.items() if k != 'result'} if 'selftest' in dir() else None,
    'raw_body_height_cm': raw_body_height_cm,
    'raw_measurements_cm': raw_measurements_cm,
    'known_height_calibration_applied': calibrated_measurements_cm is not None,
    'known_height_cm': KNOWN_HEIGHT_CM,
    'calibrated_scale_factor': calibrated_scale_factor,
    'calibrated_measurements_cm': calibrated_measurements_cm,
    'warnings': extraction['warnings'] if extraction else [],
    'failures': extraction['failures'] if extraction else [],
    'generated_at_utc': datetime.datetime.utcnow().isoformat() + 'Z',
    'note': (
        'Technical smoke test only -- NOT an anthropometric accuracy claim. A successful '
        'measurement here proves clad-body computed a circumference/length from the '
        'SAM-3D-Body-predicted mesh, not that the mesh matches the true photographed body.'
    ),
}

runtime_output_path = f'{RESULTS_DIR_RUNTIME}/measurement_output.json'
with open(runtime_output_path, 'w') as f:
    json.dump(output_record, f, indent=2)
print('Saved (runtime-only, not committed to git):', runtime_output_path)

if used_sample_image:
    repo_results_dir = f'{EXPERIMENT_DIR}/results'
    os.makedirs(repo_results_dir, exist_ok=True)
    with open(f'{repo_results_dir}/measurement_output.json', 'w') as f:
        json.dump(output_record, f, indent=2)
    print('Also written into the cloned repo (public sample image only):',
          f'{repo_results_dir}/measurement_output.json',
          '-- not committed/pushed automatically, that remains a manual step.')
else:
    print('Personal image used -- result kept in the Colab runtime only, NOT written into the cloned repo.')

## 14. Compute/runtime summary (measured, not estimated)

In [ ]:
compute_summary = {
    'environment_a': {
        'python_version': primary_telemetry.get('python_version'),
        'torch_version': primary_telemetry.get('torch_version'),
        'torch_cuda_version': primary_telemetry.get('torch_cuda_version'),
        'gpu_name': primary_telemetry.get('gpu_name'),
        'peak_vram_mb': primary_telemetry.get('peak_vram_mb'),
        'sam3d_load_time_s': primary_telemetry.get('sam3d_load_time_s'),
        'sam3d_inference_time_s': primary_telemetry.get('sam3d_inference_time_s'),
        'checkpoint_size_bytes': checkpoint_size_bytes,
        'checkpoint_download_time_s': checkpoint_download_time_s,
    },
    'environment_b': {
        'build_path': 'pixi' if env_b_python and '.pixi' in env_b_python else 'pip_fallback' if env_b_python else None,
        'measurement_extraction_time_s': None,  # captured inside measure_via_subprocess's own worker timing, see raw_result
    },
    'system_ram_gb': round(ram_gb, 1) if ram_gb else None,
}
print(json.dumps(compute_summary, indent=2))

## 15. Optional: second-checkpoint comparison (only if cheap — off by default)

Internal consistency check only (same image, two checkpoints) — **not** an accuracy comparison;
no ground truth exists here. Reuses the exact same `run_phase_a_inference` /
`run_phase_b_extraction` helpers, so nothing about the pipeline is duplicated for this.

In [ ]:
RUN_SECOND_CHECKPOINT_COMPARISON = False
SECOND_CHECKPOINT_REPO = 'facebook/sam-3d-body-vith'

if RUN_SECOND_CHECKPOINT_COMPARISON and access_status.get(SECOND_CHECKPOINT_REPO) == 'accessible' and mhr_clad_environment_ok:
    ckpt2_dir = f"/content/checkpoints/{SECOND_CHECKPOINT_REPO.split('/')[-1]}"
    snapshot_download(repo_id=SECOND_CHECKPOINT_REPO, local_dir=ckpt2_dir, token=HF_TOKEN)
    telemetry2 = run_phase_a_inference(ckpt2_dir, input_image_path, tag='secondary')
    extraction2 = None
    if telemetry2.get('interchange_path'):
        extraction2 = run_phase_b_extraction(telemetry2['interchange_path'], env_b_python)
    meas2 = extraction2['raw_result']['measurements_cm'] if extraction2 and extraction2['raw_result'] else None
    compare_keys = ['height_cm', 'bust_cm', 'waist_cm', 'hip_cm', 'shoulder_width_cm']
    print(f"{'key':16s} {PRIMARY_CHECKPOINT_REPO:28s} {SECOND_CHECKPOINT_REPO:24s}")
    for k in compare_keys:
        v1 = raw_measurements_cm.get(k) if raw_measurements_cm else None
        v2 = meas2.get(k) if meas2 else None
        print(f'{k:16s} {str(v1):28s} {str(v2):24s}')
    print('inference runtime (s):', primary_telemetry.get('sam3d_inference_time_s'), 'vs', telemetry2.get('sam3d_inference_time_s'))
    print('peak VRAM (MB):', primary_telemetry.get('peak_vram_mb'), 'vs', telemetry2.get('peak_vram_mb'))
else:
    print('Second-checkpoint comparison skipped (disabled by default, access not confirmed, or',
          'Phase B self-test did not pass). Expected -- only run this if cheap; deferred otherwise.')

## 16. Decision gate

In [ ]:
phases = phase_summary(state)
gate, reason = classify(state)

print('=== Per-phase result (Task 03B section 11) ===')
print('SAM3D_ENVIRONMENT:  ', phases['SAM3D_ENVIRONMENT'])
print('SAM3D_INFERENCE:    ', phases['SAM3D_INFERENCE'])
print('MHR_CLAD_ENVIRONMENT:', phases['MHR_CLAD_ENVIRONMENT'])
print('MHR_CLAD_EXTRACTION:', phases['MHR_CLAD_EXTRACTION'])
print('END_TO_END:         ', phases['END_TO_END'])
print('First failing boundary:', phases['first_failing_boundary'] or 'none -- all phases passed')
print()
print('=== Overall decision gate (Task 03 section 18) ===')
print('DECISION GATE:', gate.value, '-', gate.name)
print('Reason:', reason)
print()
print('Recorded failure categories:', state.failure_categories or 'none')